## Anomaly Detection — Fase 1
## Conv-AE y LSTM-AE con error por variable

Este notebook implementa el flujo completo:
1. Carga de datos SKAB
2. Entrenamiento del autoencoder (solo con datos normales)
3. Evaluación con etiquetas reales
4. Generación de la interfaz hacia el sistema experto futuro
5. Registro de todo en MLflow

### Antes de empezar
Asegúrate de haber descargado la carpeta `data/` de SKAB en `..SKAB/data/`:
```bash
svn export https://github.com/waico/SKAB/trunk/data ..SKAB/data
```

In [2]:
import sys
sys.path.append('..')

import numpy as np
import torch
import mlflow
import matplotlib.pyplot as plt

from src.data_loader import prepare_dataloaders, SENSOR_COLS
from src.models import ConvAE, LSTMAE
from src.trainer import train
from src.evaluator import compute_reconstruction_errors
from src.evaluator import (
    compute_reconstruction_errors,
    compute_threshold,
    evaluate,
    plot_reconstruction_errors,
    plot_expert_interface_sample,
)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

PyTorch version: 2.5.1
CUDA disponible: False


## 1. Configuración del experimento

Modifica estos parámetros para tus experimentos.
MLflow registrará cada combinación de forma independiente.

In [1]:
# --- Elige la arquitectura ---
ARCHITECTURE = "ConvAE"   # "ConvAE" o "LSTMAE"

# --- Hiperparámetros ---
CONFIG = {
    "run_name"                : f"{ARCHITECTURE}_w30_lr1e3",
    "architecture"            : ARCHITECTURE,
    "window_size"             : 30,    # segundos de contexto por ventana
    "step"                    : 1,     # desplazamiento entre ventanas
    "batch_size"              : 64,
    "learning_rate"           : 1e-3,
    "epochs"                  : 100,
    "early_stopping_patience" : 10,
    "threshold_percentile"    : 95,    # percentil para el umbral de anomalía
    # Específicos de ConvAE
    "latent_dim"              : 16,
    # Específicos de LSTMAE
    "hidden_dim"              : 64,
    "n_layers"                : 2,
}

DATA_DIR = "../SKAB/data/valve1"  # Ajusta si tu carpeta data está en otro lugar

## 2. Carga y preprocesado de datos

In [3]:
train_loader, val_loader, test_data, scaler, full_df = prepare_dataloaders(
    data_dir=DATA_DIR,
    window_size=CONFIG["window_size"],
    step=CONFIG["step"],
    batch_size=CONFIG["batch_size"],
)

print(f"\nVariables de sensor: {SENSOR_COLS}")
print(f"\nEjemplo de datos (primeras filas):")
full_df.head()

OK Cargados 16 ficheros - 18,160 filas totales
   Filas normales : 11,851 (65.3%)
   Filas anómalas : 6,309 (34.7%)

Ventanas - train: 10,049 | val: 1,773

Variables de sensor: ['Accelerometer1RMS', 'Accelerometer2RMS', 'Current', 'Pressure', 'Temperature', 'Thermocouple', 'Voltage', 'RateRMS']

Ejemplo de datos (primeras filas):


,Accelerometer1RMS,Accelerometer2RMS,Current,Pressure,Temperature,Thermocouple,Voltage,RateRMS,anomaly,changepoint,source_file
datetime,,,,,,,,,,,
2020-03-09 10:14:33,0.026588,0.040111,1.33020,0.054711,79.3366,26.0199,233.062,32.0,0.0,0.0,0.csv
2020-03-09 10:14:34,0.026170,0.040453,1.35399,0.382638,79.5158,26.0258,236.040,32.0,0.0,0.0,0.csv
2020-03-09 10:14:35,0.026199,0.039419,1.54006,0.710565,79.3756,26.0265,251.380,32.0,0.0,0.0,0.csv
2020-03-09 10:14:36,0.026027,0.039641,1.33458,0.382638,79.6097,26.0393,234.392,32.0,0.0,0.0,0.csv
2020-03-09 10:14:37,0.026290,0.040273,1.07851,-0.273216,79.6109,26.0420,225.342,32.0,0.0,0.0,0.csv


## 3. Definición del modelo

In [ ]:
n_features = len(SENSOR_COLS)

if ARCHITECTURE == "ConvAE":
    model = ConvAE(
        n_features=n_features,
        window_size=CONFIG["window_size"],
        latent_dim=CONFIG["latent_dim"],
    )
else:
    model = LSTMAE(
        n_features=n_features,
        window_size=CONFIG["window_size"],
        hidden_dim=CONFIG["hidden_dim"],
        n_layers=CONFIG["n_layers"],
    )

n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo: {ARCHITECTURE}")
print(f"Parámetros totales: {n_params:,}")
print(model)

## 4. Entrenamiento con MLflow

Abre una terminal y ejecuta `mlflow ui` para ver el progreso en tiempo real en http://localhost:5000

In [ ]:
model, history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=CONFIG,
    experiment_name="skab_anomaly_detection",
)

# Curvas de aprendizaje
plt.figure(figsize=(10, 4))
plt.plot(history["train_loss"], label="Train loss")
plt.plot(history["val_loss"], label="Val loss")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Curvas de aprendizaje")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Evaluación sobre el conjunto de test

El test incluye tanto datos normales como anómalos.
Usamos las etiquetas de SKAB para medir el rendimiento.

In [ ]:
# Calcular errores en train para fijar el umbral correctamente
train_windows = []
for x, _ in train_loader:
    train_windows.append(x.numpy())
train_windows = np.concatenate(train_windows, axis=0)

train_errors, _ = compute_reconstruction_errors(model, train_windows)

# Evaluación completa
results = evaluate(
    model=model,
    test_data=test_data,
    train_errors=train_errors,
    threshold_percentile=CONFIG["threshold_percentile"],
    sensor_cols=SENSOR_COLS,
)

## 6. Visualización del error de reconstrucción

In [ ]:
fig = plot_reconstruction_errors(results, SENSOR_COLS)
plt.show()

## 7. Interfaz hacia el sistema experto

Este DataFrame es la salida que consumirá el sistema experto futuro.
Cada fila = una ventana temporal.
Cada columna = score de anomalía por variable + flag binario.

In [ ]:
expert_df = results["expert_interface"]

# Mostrar muestra de ventanas que incluyan anomalías
anomaly_idx = np.where(results["labels"] == 1)[0]
if len(anomaly_idx) > 0:
    start = max(0, anomaly_idx[0] - 5)
    display(expert_df.iloc[start:start+15].round(4))

print("\nUmbrales por variable:")
for var, thr in results["threshold_per_var"].items():
    print(f"  {var:25s}: {thr:.5f}")

## 8. Ejemplo de reglas del sistema experto

Aquí puedes empezar a definir reglas con los responsables de energía.
Estas reglas operan directamente sobre el DataFrame de la interfaz.

In [ ]:
def apply_expert_rules(expert_df: "pd.DataFrame") -> "pd.DataFrame":
    """
    Ejemplo de reglas expertas sobre los scores por variable.
    
    Estas reglas son un punto de partida — deben ser refinadas
    con el conocimiento de los responsables de energía.
    """
    df = expert_df.copy()
    df["diagnosis"] = "Normal"

    # Regla 1: Fallo eléctrico en motor
    # Corriente o voltaje anómalos sin causa mecánica (vibración normal)
    mask_electrico = (
        df["flag_Current"] | df["flag_Voltage"]
    ) & ~df["flag_Accelerometer1RMS"] & ~df["flag_Accelerometer2RMS"]
    df.loc[mask_electrico, "diagnosis"] = "Posible fallo eléctrico"

    # Regla 2: Problema mecánico en bomba
    # Vibración alta + caudal o presión anómalos
    mask_mecanico = (
        df["flag_Accelerometer1RMS"] | df["flag_Accelerometer2RMS"]
    ) & (df["flag_RateRMS"] | df["flag_Pressure"])
    df.loc[mask_mecanico, "diagnosis"] = "Posible fallo mecánico en bomba"

    # Regla 3: Pérdida térmica
    # Temperatura del fluido anómala sin variación de caudal
    mask_termico = df["flag_Thermocouple"] & ~df["flag_RateRMS"]
    df.loc[mask_termico, "diagnosis"] = "Posible pérdida térmica"

    # Regla 4: Anomalía generalizada (múltiples variables)
    mask_general = df["n_vars_anomalous"] >= 4
    df.loc[mask_general, "diagnosis"] = "Anomalía generalizada — revisar sistema"

    return df


# Aplicar reglas
diagnosed_df = apply_expert_rules(expert_df)

# Resumen de diagnósticos
print("Distribución de diagnósticos:")
print(diagnosed_df["diagnosis"].value_counts())

# Mostrar ventanas con diagnóstico distinto a Normal
alerts = diagnosed_df[diagnosed_df["diagnosis"] != "Normal"]
print(f"\nTotal alertas generadas: {len(alerts)}")
alerts.head(10)

## 9. Heatmap de la interfaz con el sistema experto

In [ ]:
fig = plot_expert_interface_sample(results)
plt.show()

## 10. Guardar umbrales para el sistema experto

Los umbrales por variable son la configuración que el sistema experto
necesitará en producción.

In [ ]:
import json

thresholds_config = {
    "model"            : ARCHITECTURE,
    "window_size"      : CONFIG["window_size"],
    "global_threshold" : float(results["threshold"]),
    "per_variable"     : {k: float(v) for k, v in results["threshold_per_var"].items()},
}

with open(f"../thresholds_{ARCHITECTURE}.json", "w") as f:
    json.dump(thresholds_config, f, indent=2)

print("Umbrales guardados en thresholds.json:")
print(json.dumps(thresholds_config, indent=2))